In [1]:
def compute_first(grammar):
    first = {nt: set() for nt in grammar}

    def get_first(symbol):
        # If symbol is terminal, FIRST is the symbol itself
        if symbol not in grammar:
            return {symbol}

        res = set()
        for production in grammar[symbol]:
            if production == 'ε':
                res.add('ε')
            else:
                for char in production:
                    char_first = get_first(char)
                    res.update(char_first - {'ε'})
                    if 'ε' not in char_first:
                        break
                else:
                    res.add('ε')
        return res

    for nt in grammar:
        first[nt] = get_first(nt)
    return first

def compute_follow(grammar, first, start_symbol):
    follow = {nt: set() for nt in grammar}
    follow[start_symbol].add('$')

    # Iterate until no more changes occur
    changed = True
    while changed:
        changed = False
        for nt, productions in grammar.items():
            for prod in productions:
                if prod == 'ε': continue

                for i, char in enumerate(prod):
                    if char in grammar: # If it is a non-terminal
                        prev_len = len(follow[char])

                        # Rule: A -> a B beta, add FIRST(beta) to FOLLOW(B)
                        rest = prod[i+1:]
                        if rest:
                            # Get FIRST of the remaining string
                            first_rest = set()
                            for r_char in rest:
                                f = first[r_char] if r_char in grammar else {r_char}
                                first_rest.update(f - {'ε'})
                                if 'ε' not in f: break
                            else:
                                first_rest.add('ε')

                            follow[char].update(first_rest - {'ε'})

                            # Rule: If FIRST(beta) contains epsilon, add FOLLOW(A) to FOLLOW(B)
                            if 'ε' in first_rest:
                                follow[char].update(follow[nt])
                        else:
                            # Rule: A -> a B, add FOLLOW(A) to FOLLOW(B)
                            follow[char].update(follow[nt])

                        if len(follow[char]) > prev_len:
                            changed = True
    return follow

# --- Test Case ---
# E -> TX, X -> +TX | ε, T -> FY, Y -> *FY | ε, F -> (E) | i
cfg = {
    'E': ['TX'],
    'X': ['+TX', 'ε'],
    'T': ['FY'],
    'Y': ['*FY', 'ε'],
    'F': ['(E)', 'i']
}

first_sets = compute_first(cfg)
follow_sets = compute_follow(cfg, first_sets, 'E')

print("Non-Terminal | FIRST Set       | FOLLOW Set")
print("-" * 45)
for nt in cfg:
    print(f"{nt:<12} | {str(sorted(list(first_sets[nt]))):<15} | {str(sorted(list(follow_sets[nt])))}")

Non-Terminal | FIRST Set       | FOLLOW Set
---------------------------------------------
E            | ['(', 'i']      | ['$', ')']
X            | ['+', 'ε']      | ['$', ')']
T            | ['(', 'i']      | ['$', ')', '+']
Y            | ['*', 'ε']      | ['$', ')', '+']
F            | ['(', 'i']      | ['$', ')', '*', '+']
